## Check Pytorch Availability

### Import PyTorch
Loads the deep-learning library.

In [1]:
import torch

### Check CUDA availability
Can PyTorch see the GPU? → `True`.

In [2]:
torch.cuda.is_available()

True

### Check PyTorch build & GPU driver
Confirms the CUDA build of PyTorch and that the NVIDIA driver responds.

In [3]:
import torch
import subprocess

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA build: {torch.version.cuda}")

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print("Driver status: OK (nvidia-smi works)") if result.returncode == 0 else print("Driver status: FAILED")
except FileNotFoundError:
    print("Driver status: nvidia-smi NOT FOUND")


PyTorch version: 2.14.0+cu130
PyTorch CUDA build: 13.0
Driver status: OK (nvidia-smi works)


### Show GPU name
Prints which GPU PyTorch will use → `NVIDIA GeForce RTX 4060`.

In [4]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 4060'

### Load datasets

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### Load, split & scale the dataset
569 patients × 30 features → 80/20 train/test split → each feature scaled to mean 0, std 1.

In [6]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Inspect scaled features
Peek at the scaled training data → `(455, 30)`, values centred around 0.

In [7]:
X_train_scaled

array([[ 0.04156308, -0.58461567, -0.06980235, ..., -1.21743811,
        -0.74493481, -1.23953347],
       [-0.84131542, -0.63093537, -0.8754438 , ..., -1.10998941,
        -0.87887353, -0.70151638],
       [ 1.03619836, -1.41605425,  0.91744361, ..., -0.46061238,
        -1.95038333, -1.37864692],
       ...,
       [-0.6792681 , -0.61935544, -0.70277733, ..., -0.66810474,
        -2.07717865, -0.55625735],
       [ 2.07274243,  0.70538792,  2.03063452, ...,  1.62776119,
        -1.45570297, -0.68363834],
       [-0.62897756, -0.45955248, -0.64264877, ..., -0.36918275,
        -1.16996702, -1.28813937]], shape=(455, 30))

### Inspect labels
Peek at the training labels → 0 = malignant, 1 = benign.

In [8]:
y_train

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1,
       1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1,

### Convert to tensors
NumPy arrays → PyTorch tensors; labels reshaped to `[n, 1]` to match the model output.

In [9]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

### Pair features with labels
Bundles each patient's 30 features with their label.

In [10]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

### Count input features
Number of features per patient (30) → the input size of the first layer.

In [11]:

instanceNo = X_train_scaled_tensor.shape[1]

# result: torch.Size([455, 30])
# meaning there are 30 features per instance

### Batch the training data
Serves shuffled batches of 32 patients to the training loop.

In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

### Build Neural Network

In [13]:
# This is the structure on how to define a class in Pytorch
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()

        # The 2nd param of `nn.Linear` is the output number 
        # for the next layer
        self.fc1 = nn.Linear(instanceNo, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        # Drop out layer is optional
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Drop out layer is optional
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x)) # sigmoid is to get value between 0 & 1
        
        return x

### Create model, loss & optimizer
BCE loss = how wrong a prediction is; Adam = how weights get adjusted (lr 0.001).

In [14]:
# Define a loss function and optimizer (learning rate)
# Since it's binary classification, we can choose Binary Cross Entropy Loss
criterion = nn.BCELoss()

# Define a model optimizer (including learning rate)
# Since it's binary classification, we can choose Binary Cross Entropy Loss
model = BCNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Train Neural Network

In [15]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss was {running_loss/len(train_loader)}')

Epoch 1: Loss was 0.643987754980723
Epoch 2: Loss was 0.5127697666486104
Epoch 3: Loss was 0.34858172039190927
Epoch 4: Loss was 0.22731209695339202
Epoch 5: Loss was 0.15518508305152257
Epoch 6: Loss was 0.11951137632131577
Epoch 7: Loss was 0.10181132405996322
Epoch 8: Loss was 0.08540193761388461
Epoch 9: Loss was 0.0788286526997884
Epoch 10: Loss was 0.07380026786898573
Epoch 11: Loss was 0.06514860580985744
Epoch 12: Loss was 0.05690389772256215
Epoch 13: Loss was 0.05850344529996316
Epoch 14: Loss was 0.06266678140188257
Epoch 15: Loss was 0.05305019191776713
Epoch 16: Loss was 0.07631049898142615
Epoch 17: Loss was 0.05101135063450783
Epoch 18: Loss was 0.07111227878679832
Epoch 19: Loss was 0.0453701081375281
Epoch 20: Loss was 0.04493474874955913


### Evaluate test accuracy
Checks predictions on unseen patients → ~97–98% accuracy.

In [16]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()
    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

accuracy

0.9649122953414917

# Phase 2: Export to ONNX

### Prepare model for export
`eval()` switches Dropout off (deterministic, as a circuit needs); CPU for a device-independent export.

In [17]:
import os

os.makedirs("artifacts", exist_ok=True)   # folder for all ZK files

model.eval()        # inference mode: Dropout switches OFF
model.to("cpu")     # export from CPU so the result doesn't depend on the GPU


BCNet(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

### Pick the sample patient
Test patient #0, shape `[1, 30]`: the input we export with and later prove.

In [18]:
sample_input = X_test_scaled_tensor[0:1]   # first test patient; [0:1] keeps the batch dimension
sample_input.shape                         # torch.Size([1, 30])

torch.Size([1, 30])

### Export to ONNX
Records the model's operations + weights into `artifacts/network.onnx` (opset 18 for EZKL).

In [19]:
onnx_program = torch.onnx.export(
    model,
    (sample_input,),                 # example inputs go in a tuple
    input_names=["input"],
    output_names=["output"],
    opset_version=18,               # the newest version EZKL is tested with
    dynamo=True,                    # the official exporter (the default; written out to be clear)
)

onnx_program.save("artifacts/network.onnx")

W0927 18:04:31.660000 18136 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0927 18:04:31.661000 18136 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W0927 18:04:31.662000 18136 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W0927 18:04:31.663000 18136 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d
W0927 18:04:31.748000 18136 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `BCNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BCNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


### Check the ONNX graph
Validates the file and lists its ops → `Gemm, Relu, Gemm, Relu, Gemm, Sigmoid`.

In [20]:
import onnx

graph = onnx.load("artifacts/network.onnx")
onnx.checker.check_model(graph)

print("Operations:", [node.op_type for node in graph.graph.node])
print("Opset:", graph.opset_import[0].version)
print("File size:", os.path.getsize("artifacts/network.onnx"), "bytes")


Operations: ['Gemm', 'Relu', 'Gemm', 'Relu', 'Gemm', 'Sigmoid']
Opset: 18
File size: 22922 bytes


### Save `input.json`
Writes the sample patient's 30 values in EZKL's input format.

In [21]:
# Save `input.json` for EZKL inference
import json

input_data = {"input_data": [sample_input[0].tolist()]}

with open("artifacts/input.json", "w") as f:
    json.dump(input_data, f)

print("Features:", len(input_data["input_data"][0]))
print("First 3 values:", input_data["input_data"][0][:3])


Features: 30
First 3 values: [-0.33282211422920227, -0.7698944807052612, -0.36069455742836]


### Record PyTorch reference prediction
The float answer to compare against the circuit's integer answer later.

In [22]:
# Record what PyTorch predicts for this patient
with torch.no_grad():
    pytorch_output = model(sample_input).item()

print("PyTorch prediction:", pytorch_output)
print("True label:", y_test[0])


PyTorch prediction: 0.9978439807891846
True label: 1


# Phase 3: EZKL local proof

### Generate default settings
Visibility: inputs/outputs public, weights fixed → default `settings.json` (scale 7, logrows 17).

In [23]:
# Generate the default settings

import ezkl

MODEL    = "artifacts/network.onnx"
INPUT    = "artifacts/input.json"
CALIB    = "artifacts/calibration.json"
SETTINGS = "artifacts/settings.json"

run_args = ezkl.PyRunArgs()
run_args.input_visibility  = "public"   # verifier sees the patient's 30 features
run_args.output_visibility = "public"   # verifier sees the prediction
run_args.param_visibility  = "fixed"    # weights built into the circuit: hidden but bound

ezkl.gen_settings(MODEL, SETTINGS, py_run_args=run_args)

s = json.load(open(SETTINGS))["run_args"]
print("Defaults -> scale:", s["input_scale"], "| logrows:", s["logrows"], "| lookup_range:", s["lookup_range"])


Defaults -> scale: 7 | logrows: 17 | lookup_range: [-32768, 32768]


### Build calibration data
50 real training patients → `calibration.json`, so EZKL sees the real value range.

In [24]:
# Build the calibration data for the circuit

calib_data = {"input_data": [X_train_scaled[:50].flatten().tolist()]}

with open(CALIB, "w") as f:
    json.dump(calib_data, f)

print("Values in calibration file:", len(calib_data["input_data"][0]))   # 50 × 30 = 1500


Values in calibration file: 1500


### Calibrate scale & logrows
Picks the precision (scale 11 = 1/2048 steps) and circuit size (logrows 18) from real data. Red error lines are normal.

In [25]:
# Calibrate the circuit settings based on the calibration data

ok = ezkl.calibrate_settings(CALIB, MODEL, SETTINGS, "resources")
print("Calibration OK:", ok)

s = json.load(open(SETTINGS))["run_args"]
print("Calibrated -> scale:", s["input_scale"], "| logrows:", s["logrows"], "| lookup_range:", s["lookup_range"])


[tensor] decomposition error: integer 18366412826 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer -44508310199 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 293842964455 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 139792090524 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer -126514569084 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 1175413427290 is too large to be represented by base 16384 and n 2
forwa

Calibration OK: True
Calibrated -> scale: 11 | logrows: 18 | lookup_range: [-83050, 56052]


### Compile the circuit
ONNX + settings → `network.compiled` (the circuit, weights baked in).

In [26]:
# Compile
COMPILED = "artifacts/network.compiled"

ezkl.compile_circuit(MODEL, COMPILED, SETTINGS)
print("Compiled:", os.path.getsize(COMPILED), "bytes")


Compiled: 135316 bytes


### Resume cell (after kernel restart)
Re-creates imports, paths and `ensure_srs()` without retraining; sets `HOME`, which EZKL needs on Windows.

In [27]:
# Reread the settings to get the logrows for the SRS. Useful when the kernel is restarted and the SRS is already cached.
import os, sys, json, subprocess, ezkl
os.environ.setdefault("HOME", os.path.expanduser("~"))   # Windows has no HOME; EZKL needs it

MODEL, INPUT, CALIB, SETTINGS = ("artifacts/network.onnx", "artifacts/input.json",
                                 "artifacts/calibration.json", "artifacts/settings.json")
COMPILED = "artifacts/network.compiled"

def ensure_srs(settings_path):
    """Local SRS path for this circuit's logrows; downloads it once if missing."""
    logrows = json.load(open(settings_path))["run_args"]["logrows"]
    srs = os.path.expanduser(f"~/.ezkl/srs/kzg{logrows}.srs")
    if not os.path.exists(srs):   # child process: an EZKL panic can't poison this kernel
        r = subprocess.run([sys.executable, "-c", "import asyncio, ezkl\n"
                            f"async def main(): return await ezkl.get_srs({settings_path!r})\n"
                            "print(asyncio.run(main()))"],
                           capture_output=True, text=True)
        if not os.path.exists(srs):
            raise RuntimeError("SRS download failed: " + r.stderr[-500:])
    return srs

### Get the SRS
Public KZG setup file (like circom's `.ptau`), sized to logrows; downloaded only if missing.

In [28]:
# Get the SRS: download only if it's not cached yet
SRS = ensure_srs(SETTINGS)
print("SRS:", SRS, os.path.getsize(SRS), "bytes")

SRS: C:\Users\rasyi/.ezkl/srs/kzg18.srs 33554692 bytes


### Generate proving & verifying keys
Circuit + SRS → `pk.key` (1.3 GB, prover) and `vk.key` (0.6 MB, verifier; commits to the weights).

In [29]:
# Generate the keys
VK = "artifacts/vk.key"
PK = "artifacts/pk.key"
ezkl.setup(COMPILED, VK, PK, srs_path=SRS)

print("Verifying key:", os.path.getsize(VK), "bytes")
print("Proving key:  ", os.path.getsize(PK), "bytes")


Verifying key: 591367 bytes
Proving key:   1309214827 bytes


### Generate the witness
Runs the patient through the circuit in integer math → output `1/2048` ≈ PyTorch's 0.000489.

In [30]:
# Generate the witness
WITNESS = "artifacts/witness.json"

ezkl.gen_witness(INPUT, COMPILED, WITNESS)

w = json.load(open(WITNESS))
print("Circuit output:", w["pretty_elements"]["rescaled_outputs"])
print("Raw field element:", w["outputs"])


Circuit output: [['0.998046875']]
Raw field element: [['fc07000000000000000000000000000000000000000000000000000000000000']]


### Generate the proof
Witness + proving key → `proof.json` (~25 KB, ~9 s).

In [31]:
# Generate the proof
PROOF = "artifacts/proof.json"

ezkl.prove(WITNESS, COMPILED, PK, PROOF, srs_path=SRS)
print("Proof size:", os.path.getsize(PROOF), "bytes")


Proof size: 25084 bytes


### Verify the proof locally
Checks the proof with the verifying key → `True` (tampered → `False`).

In [32]:
# Verify the proof locally
try:
    ok = ezkl.verify(PROOF, SETTINGS, VK, srs_path=SRS)
except RuntimeError as e:
    ok = False
    print("Verification failed:", e)

print("Proof valid:", ok)


Proof valid: True


# Phase 4: Verify on Chain

### Generate the Solidity verifier
Turns `vk.key` into `Verifier.sol` (`Halo2Verifier.verifyProof(bytes, uint256[])`).

In [33]:
SOL = "artifacts/Verifier.sol"
ABI = "artifacts/Verifier.abi"

await ezkl.create_evm_verifier(VK, SETTINGS, SOL, ABI, srs_path=SRS)

print("Verifier.sol:", os.path.getsize(SOL), "bytes")
print(open(ABI).read())


Verifier.sol: 81096 bytes
[{"type":"function","name":"verifyProof","inputs":[{"name":"proof","type":"bytes","internalType":"bytes"},{"name":"instances","type":"uint256[]","internalType":"uint256[]"}],"outputs":[{"name":"","type":"bool","internalType":"bool"}],"stateMutability":"nonpayable"}]


### Foundry setup
Paths, local RPC, Anvil's dev key, the `run()` helper, and `verify_onchain()` (deploy → verify → size/gas → tamper test on a fresh chain).

In [34]:
# Foundry setup: local chain settings and helpers
import shutil, time

FOUNDRY = os.path.expanduser("~/.foundry/bin")
RPC = "http://127.0.0.1:8545"
KEY = "0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80"   # Anvil's public dev account #0

def run(*cmd, cwd=None):
    r = subprocess.run([os.path.join(FOUNDRY, cmd[0]), *cmd[1:]], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr.strip() or r.stdout.strip())
    return r.stdout.strip()

def verify_onchain(sol_path, proof_path, calldata_path, contract_file):
    """Fresh Anvil chain: deploy the verifier, verify the proof, measure size + gas, tamper test."""
    shutil.copy(sol_path, f"contracts/src/{contract_file}")
    calldata = ezkl.encode_evm_calldata(proof_path, calldata_path)
    anvil = subprocess.Popen([os.path.join(FOUNDRY, "anvil"), "--port", "8545"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    try:
        out = run("forge", "create", f"src/{contract_file}:Halo2Verifier", "--rpc-url", RPC,
                  "--private-key", KEY, "--broadcast", "--json", cwd="contracts")
        addr = json.loads(out)["deployedTo"]

        def accepts(data):   # the verifier returns true, or reverts on a bad proof
            try:
                return int(run("cast", "call", addr, "--data", "0x" + data.hex(), "--rpc-url", RPC), 16) == 1
            except RuntimeError:
                return False

        # Tamper test: flip the last byte of the output instance
        inst = int.from_bytes(calldata[36:68], "big")               # where the instances array starts
        n = int.from_bytes(calldata[4 + inst:4 + inst + 32], "big")  # 30 inputs + 1 output
        bad = bytearray(calldata)
        bad[4 + inst + 32 * n + 31] ^= 1

        return {"valid": accepts(calldata),
                "tampered_valid": accepts(bad),
                "runtime_bytes": int(run("cast", "codesize", addr, "--rpc-url", RPC)),
                "gas": int(run("cast", "estimate", addr, "0x" + calldata.hex(), "--rpc-url", RPC))}
    finally:
        anvil.terminate()   # always stop the local chain, even if something fails

### Build the verifier contract
Copies the latest `Verifier.sol` into `contracts/` and compiles it → 15,194-byte runtime (< 24 KB limit).

In [35]:
# Build the verifier contract with Foundry
shutil.copy(SOL, "contracts/src/Verifier.sol")
print(run("forge", "build", "--sizes", cwd="contracts"))


Compiling 22 files with Solc 0.8.20
Solc 0.8.20 finished in 1.02s
Compiler run successful with warnings:
Warning (2018): Function state mutability can be restricted to view
  --> src/Verifier.sol:65:5:
   |
65 |     function verifyProof(
   |     ^ (Relevant source part starts here and spans across multiple lines).

Warning (2018): Function state mutability can be restricted to view
  --> src/VerifierLogits.sol:65:5:
   |
65 |     function verifyProof(
   |     ^ (Relevant source part starts here and spans across multiple lines).


â•­----------------------------------------+------------------+-------------------+--------------------+---------------------â•®
| Contract                               | Runtime Size (B) | Initcode Size (B) | Runtime Margin (B) | Initcode Margin (B) |
+==========================================================================================================================+
| Halo2Verifier (src/Verifier.sol)       | 15,194           | 15,223            | 9

### Verify on-chain
Fresh Anvil chain → deploy `Halo2Verifier` → `verifyProof` → size & gas → tamper test → stop the chain.
Expected: valid `True`, tampered `False` (the verifier reverts), 15,194-byte runtime, ~639k gas.

In [36]:
# Deploy, verify, measure gas and tamper-test on a fresh local chain
verify_onchain(SOL, PROOF, "artifacts/calldata.bytes", "Verifier.sol")

{'valid': True, 'tampered_valid': False, 'runtime_bytes': 15194, 'gas': 644495}

# Experiment: Same Model, but Without Sigmoid

In [37]:
# Experiment: same model, without Sigmoid
import os, onnx

os.makedirs("artifacts_logits", exist_ok=True)
MODEL_L = "artifacts_logits/network.onnx"

g = onnx.load("artifacts/network.onnx")
logit = g.graph.node[-1].input[0]   # what goes into Sigmoid = the raw score

onnx.utils.extract_model("artifacts/network.onnx", MODEL_L, ["input"], [logit])

print("Ops:", [n.op_type for n in onnx.load(MODEL_L).graph.node])


Ops: ['Gemm', 'Relu', 'Gemm', 'Relu', 'Gemm']


In [38]:
# Settings + calibration for the no-Sigmoid model
import json, ezkl
os.environ.setdefault("HOME", os.path.expanduser("~"))

SETTINGS_L = "artifacts_logits/settings.json"

run_args = ezkl.PyRunArgs()
run_args.input_visibility  = "public"
run_args.output_visibility = "public"
run_args.param_visibility  = "fixed"

ezkl.gen_settings(MODEL_L, SETTINGS_L, py_run_args=run_args)
ok = ezkl.calibrate_settings("artifacts/calibration.json", MODEL_L, SETTINGS_L, "resources")
print("Calibration OK:", ok)

for name, path in [("with Sigmoid", "artifacts/settings.json"), ("no Sigmoid  ", SETTINGS_L)]:
    s = json.load(open(path))["run_args"]
    print(name, "-> scale:", s["input_scale"], "| logrows:", s["logrows"], "| lookup_range:", s["lookup_range"])


[tensor] decomposition error: integer 17472798959 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer -26744517203 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer -107071039239 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer -53498769808 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 587718258152 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 1117935407513 is too large to be represented by base 16384 and n 2
forwa

Calibration OK: True
with Sigmoid -> scale: 11 | logrows: 18 | lookup_range: [-83050, 56052]
no Sigmoid   -> scale: 13 | logrows: 15 | lookup_range: [0, 0]


In [39]:
# Compile, keys, proof, verify for the no-Sigmoid model
import time, math

COMPILED_L = "artifacts_logits/network.compiled"
VK_L, PK_L = "artifacts_logits/vk.key", "artifacts_logits/pk.key"
WITNESS_L, PROOF_L = "artifacts_logits/witness.json", "artifacts_logits/proof.json"

ezkl.compile_circuit(MODEL_L, COMPILED_L, SETTINGS_L)
SRS_L = ensure_srs(SETTINGS_L)

t = time.perf_counter(); ezkl.setup(COMPILED_L, VK_L, PK_L, srs_path=SRS_L); t_setup = time.perf_counter() - t
ezkl.gen_witness("artifacts/input.json", COMPILED_L, WITNESS_L)
t = time.perf_counter(); ezkl.prove(WITNESS_L, COMPILED_L, PK_L, PROOF_L, srs_path=SRS_L); t_prove = time.perf_counter() - t
ok = ezkl.verify(PROOF_L, SETTINGS_L, VK_L, srs_path=SRS_L)

logit = float(json.load(open(WITNESS_L))["pretty_elements"]["rescaled_outputs"][0][0])
print("Verify:", ok)
print(f"Setup {t_setup:.1f}s | Prove {t_prove:.1f}s")
print("SRS:", os.path.getsize(SRS_L), "| pk:", os.path.getsize(PK_L),
      "| vk:", os.path.getsize(VK_L), "| proof:", os.path.getsize(PROOF_L))
print("Circuit logit:", logit, "-> sigmoid outside:", 1 / (1 + math.exp(-logit)))

Verify: True
Setup 0.9s | Prove 1.0s
SRS: 4194564 | pk: 138479371 | vk: 66823 | proof: 21989
Circuit logit: 6.137939453125 -> sigmoid outside: 0.9978452843079154


In [40]:
# Solidity verifier + on-chain check for the no-Sigmoid model
SOL_L, ABI_L = "artifacts_logits/Verifier.sol", "artifacts_logits/Verifier.abi"
await ezkl.create_evm_verifier(VK_L, SETTINGS_L, SOL_L, ABI_L, srs_path=SRS_L)

# separate contract file, so tests/CI keep using Verifier.sol
# with Sigmoid: runtime 15,194 bytes, gas 639,011
verify_onchain(SOL_L, PROOF_L, "artifacts_logits/calldata.bytes", "VerifierLogits.sol")

{'valid': True, 'tampered_valid': False, 'runtime_bytes': 13426, 'gas': 575601}